# Task 2: Advanced EDA and Text Mining


**Vehicle Infotainment / Radio Warranty Claims — Exploratory Data Analysis, Text Mining, Entity Tagging & Clustering**


## Deliverable Overview


This notebook performs the full Task 2 workflow on a dataset of 1,000 vehicle warranty claims:


- **Step 1:** Exploratory Data Analysis (schema, data quality, cleaning, statistics, visualizations)


- **Step 2:** Text Mining — identify free-text fields, extract structured entity tags, classify issue types


- **Step 3:** Clustering / Topic Modelling (TF-IDF + K-Means) + root-cause identification


- **Step 4:** Stakeholder insights & actionable recommendations


All outputs (cleaned/tagged dataset, charts, report) are written to the working directory.


> **Note on NLP/LLM:** No external LLM API key is available in this environment, so entity extraction uses robust open-source NLP (NLTK tokenization + TF-IDF keyword scoring + rule-based keyword/regex dictionaries). An open-source LLM (e.g., via `transformers`/Ollama) can be dropped into the `extract_entities()` function in place of the rule-based tagger with identical output columns.

In [1]:

import os, re, json, warnings, datetime as dt
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Wordcloud (optional)
try:
    from wordcloud import WordCloud
    HAS_WORDCLOUD = True
except Exception:
    HAS_WORDCLOUD = False

# Download NLTK data (quiet)
for pkg in ["punkt", "stopwords", "wordnet", "omw-1.4"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

plt.rcParams["figure.figsize"] = (10, 6)
sns.set_style("whitegrid")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

print("Setup complete. Libraries loaded.")


ModuleNotFoundError: No module named 'numpy'

## Step 1 — Exploratory Data Analysis (EDA)


### 1.1 Load the dataset

In [ ]:

DATA_PATH = r"C:\Downloads\ Hanuai task2 assignment.csv"
OUT_DIR = r"C:\Downloads\hanu ai\task2"
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print("Loaded dataset shape:", df.shape)
print("\nFirst 3 rows:\n")
print(df.head(3).to_string())


### 1.2 Schema, data types & volume per column

In [ ]:

info = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "null_count": df.isna().sum(),
    "null_%": (df.isna().mean()*100).round(2),
    "n_unique": df.nunique(dropna=True),
    "example": [df[c].dropna().astype(str).iloc[0] if df[c].notna().any() else "" for c in df.columns]
})
print(info.to_string(index=False))
print("\nTotal rows:", len(df), "| Total columns:", df.shape[1])
info.to_csv(os.path.join(OUT_DIR, "eda_schema_summary.csv"), index=False)
print("\nSaved: eda_schema_summary.csv")


### 1.3 Missing values & data quality

In [ ]:

nulls = df.isna().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)
print("Columns with missing values:")
print(nulls.to_string())
print(f"\nTotal missing cells: {df.isna().sum().sum()}")

# Identify the free-text columns and their fill rates
text_cols = ["CAUSAL_VERBATIM", "CORRECTION_VERBATIM", "CUSTOMER_VERBATIM", "Additional Context"]
print("\nFree-text column fill rates (% non-null):")
for c in text_cols:
    print(f"  {c}: {df[c].notna().mean()*100:.1f}%")


### 1.4 Duplicates & inconsistencies

In [ ]:

print("Duplicate Event id:", df["Event id"].duplicated().sum())
print("Duplicate full rows:", df.duplicated().sum())
print("Unique Event id:", df["Event id"].nunique(), "of", len(df))

# Inconsistency checks on date fields
for c in ["Opened date", "BUILD_DATE", "IN_USE_DATE"]:
    d = pd.to_datetime(df[c], errors="coerce")
    print(f"{c}: parsed OK={d.notna().sum()}, unparseable={d.isna().sum()}, "
          f"range={d.min()} to {d.max()}")

# Check MODLYR values
print("\nMODLYR values:", sorted(df["MODLYR"].dropna().unique()))
print("MAKE values:", df["MAKE"].value_counts(dropna=False).to_dict())


### 1.5 Data cleaning

In [ ]:

df_clean = df.copy()

# --- Parse dates (normalize all to tz-naive to allow subtraction) ---
def parse_date_naive(series):
    d = pd.to_datetime(series, errors="coerce", utc=True)  # parse as UTC
    d = d.dt.tz_convert("UTC").dt.tz_localize(None)         # drop tz info
    return d

date_cols = ["Opened date", "BUILD_DATE", "IN_USE_DATE"]
for c in date_cols:
    df_clean[c + "_parsed"] = parse_date_naive(df_clean[c])

# --- Missing value handling ---
# CAUSAL_VERBATIM: 1 missing -> fill with empty string (text, cannot meaningfully impute)
df_clean["CAUSAL_VERBATIM"] = df_clean["CAUSAL_VERBATIM"].fillna("")

# CAUSAL_CD_DESC: 3 missing -> mode imputation (categorical root-cause code)
mode_causal = df_clean["CAUSAL_CD_DESC"].mode()[0]
df_clean["CAUSAL_CD_DESC"] = df_clean["CAUSAL_CD_DESC"].fillna(mode_causal)
print(f"CAUSAL_CD_DESC missing filled with mode: '{mode_causal}'")

# IN_USE_DATE: 1 missing -> leave as NaT (date, no safe impute); flag it
df_clean["IN_USE_DATE_missing"] = df_clean["IN_USE_DATE_parsed"].isna()

# Derived: days between build and opened (warranty claim lag)
df_clean["days_build_to_opened"] = (df_clean["Opened date_parsed"] - df_clean["BUILD_DATE_parsed"]).dt.days
df_clean["days_inuse_to_opened"] = (df_clean["Opened date_parsed"] - df_clean["IN_USE_DATE_parsed"]).dt.days

# Derived: opened year-month for trend analysis
df_clean["opened_ym"] = df_clean["Opened date_parsed"].dt.to_period("M").astype(str)

# Combined free-text field for mining (diagnosis + correction + customer complaint)
df_clean["verbatim_combined"] = (
    df_clean["CAUSAL_VERBATIM"].fillna("").astype(str) + " || " +
    df_clean["CORRECTION_VERBATIM"].fillna("").astype(str) + " || " +
    df_clean["CUSTOMER_VERBATIM"].fillna("").astype(str)
)

print("\nCleaning complete. Shape:", df_clean.shape)
print("New derived columns: days_build_to_opened, days_inuse_to_opened, opened_ym, verbatim_combined")
print("\nMissing after cleaning (key columns):")
print(df_clean[["CAUSAL_VERBATIM","CAUSAL_CD_DESC","IN_USE_DATE_parsed"]].isna().sum())


### 1.6 Critical columns & summary statistics

Critical stakeholder columns: `CAUSAL_CD_DESC` (root cause), `COMPLAINT_CD_DESC` (complaint type), `MAKE`, `PLANT`, `BUILD_PLANT_DESC`, `Opened date` (timing), and the free-text verbatims.


In [ ]:

# Categorical frequency summaries
cat_cols = ["MAKE", "PLANT", "BUILD_PLANT_DESC", "CAUSAL_CD_DESC", "COMPLAINT_CD_DESC", "Trigger"]
print("=== TOP VALUES FOR CRITICAL CATEGORICAL COLUMNS ===\n")
for c in cat_cols:
    print(f"--- {c} ---")
    print(df_clean[c].value_counts(dropna=False).head(10).to_string())
    print()

# Numerical summary for derived date-lag features
num_cols = ["days_build_to_opened", "days_inuse_to_opened"]
print("=== NUMERICAL SUMMARY (derived date-lag features) ===\n")
print(df_clean[num_cols].describe().to_string())
print()
# Save categorical freq
freq_rows = []
for c in cat_cols:
    vc = df_clean[c].value_counts(dropna=False)
    for val, cnt in vc.items():
        freq_rows.append({"column": c, "value": str(val), "count": int(cnt),
                          "pct": round(cnt/len(df_clean)*100, 2)})
pd.DataFrame(freq_rows).to_csv(os.path.join(OUT_DIR, "eda_categorical_frequencies.csv"), index=False)
print("Saved: eda_categorical_frequencies.csv")


### 1.7 Distribution visualizations for critical columns

In [ ]:

chart_dir = os.path.join(OUT_DIR, "charts")
os.makedirs(chart_dir, exist_ok=True)

def savefig(name):
    plt.tight_layout()
    plt.savefig(os.path.join(chart_dir, name), dpi=150, bbox_inches="tight")
    plt.close()
    print("Saved chart:", name)

# 1. Complaint type distribution
plt.figure()
df_clean["COMPLAINT_CD_DESC"].value_counts().plot(kind="bar", color="#4c72b0")
plt.title("Complaint Type Distribution (COMPLAINT_CD_DESC)")
plt.xlabel("Complaint Type"); plt.ylabel("Number of Claims")
plt.xticks(rotation=45, ha="right")
savefig("01_complaint_type_distribution.png")

# 2. Root cause distribution (top 12)
plt.figure()
top_causal = df_clean["CAUSAL_CD_DESC"].value_counts().head(12)
top_causal.plot(kind="barh", color="#55a868")
plt.title("Top Root-Cause Codes (CAUSAL_CD_DESC)")
plt.xlabel("Number of Claims"); plt.ylabel("Root Cause")
savefig("02_root_cause_distribution.png")

# 3. Claims by make
plt.figure()
df_clean["MAKE"].value_counts().plot(kind="bar", color="#dd8452")
plt.title("Claims by Vehicle Make (MAKE)")
plt.xlabel("Make"); plt.ylabel("Number of Claims")
savefig("03_claims_by_make.png")

# 4. Claims by build plant
plt.figure()
df_clean["BUILD_PLANT_DESC"].value_counts().plot(kind="bar", color="#8172b3")
plt.title("Claims by Build Plant (BUILD_PLANT_DESC)")
plt.xlabel("Build Plant"); plt.ylabel("Number of Claims")
plt.xticks(rotation=45, ha="right")
savefig("04_claims_by_build_plant.png")

# 5. Monthly claim trend
plt.figure()
monthly = df_clean.groupby("opened_ym").size()
monthly.plot(kind="line", marker="o", color="#c49c94")
plt.title("Monthly Warranty Claim Trend (Opened date)")
plt.xlabel("Year-Month"); plt.ylabel("Number of Claims")
plt.xticks(rotation=45, ha="right")
savefig("05_monthly_claim_trend.png")

# 6. Days build->opened histogram
plt.figure()
sns.histplot(df_clean["days_build_to_opened"].dropna(), bins=30, kde=True, color="#64b5cd")
plt.title("Lag: Build Date to Opened Date (days)")
plt.xlabel("Days"); plt.ylabel("Frequency")
savefig("06_lag_build_to_opened.png")

print("\nAll charts saved to:", chart_dir)


## Step 2 — Text Mining & Entity Tagging


### 2.1 Identify free-text fields

In [ ]:

print("Free-text (unstructured narrative) columns identified:")
for c in text_cols:
    nonnull = int(df_clean[c].notna().sum())
    avg_len = df_clean[c].fillna("").astype(str).str.len().mean()
    print(f"  - {c}: {nonnull}/{len(df_clean)} non-null, avg {avg_len:.0f} chars")
print("\nSelected for mining: CAUSAL_VERBATIM (technician diagnosis), "
      "CORRECTION_VERBATIM (correction action), CUSTOMER_VERBATIM (customer complaint).")
print("Additional Context is ~99.5% empty and excluded from mining.")


### 2.2 Text preprocessing

In [ ]:

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)   # keep letters only
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return " ".join(tokens)

df_clean["verbatim_clean"] = df_clean["verbatim_combined"].apply(preprocess_text)
print("Preprocessing complete. Sample cleaned text:\n")
print(df_clean["verbatim_clean"].iloc[0][:400])


### 2.3 Entity tagging — extract structured tags from free text


Rule-based NLP tagger using keyword dictionaries built around the vehicle infotainment domain (components, symptoms, error codes, actions). An open-source LLM can replace this block.


In [ ]:

# Domain keyword dictionaries (lowercase, lemmatized forms)
COMPONENT_KEYWORDS = {
    "radio": ["radio", "head unit", "infotainment", "module", "tcicm", "telematics"],
    "display": ["display", "screen", "monitor", "panel", "bezel"],
    "amplifier": ["amplifier", "amp"],
    "antenna": ["antenna", "coaxial", "coax"],
    "sd_card": ["sd card", "card reader", "receptacle"],
    "usb": ["usb", "port", "cable"],
    "speaker": ["speaker", "audio", "chime"],
    "wiring": ["wiring", "harness", "circuit", "connector", "pin", "terminal"],
    "bcm": ["bcm", "body control module"],
    "cluster": ["instrument cluster", "dic", "gauge"],
    "bluetooth": ["bluetooth", "phone"],
    "onstar": ["onstar"],
    "navigation": ["navigation", "nav", "gps", "map", "android auto", "carplay", "apple car play"],
    "camera": ["camera", "backup camera", "rear view camera"],
}

SYMPTOM_KEYWORDS = {
    "black_screen": ["black screen", "blank screen", "screen goes black", "display blank", "inop", "inoperative"],
    "no_sound": ["no sound", "no audio", "audio cut out", "no sound", "mute"],
    "freezing": ["freezing", "freeze", "locks up", "lock up", "unresponsive", "slow"],
    "intermittent": ["intermittent", "comes and goes", "occasionally", "random"],
    "loss_communication": ["loss communication", "lost communication", "no comm", "no communication", "ethernet"],
    "shorted": ["shorted", "short", "open circuit", "open condition"],
    "registers_incorrectly": ["registers incorrectly", "demo mode", "wrong channel", "wrong station"],
    "wont_load": ["wont load", "will not load", "loading", "checking map data", "initializing"],
    "connection": ["connection failure", "disconnect", "wont connect", "will not connect", "pairing"],
    "blank_on_reverse": ["reverse", "back up camera", "reversing"],
}

ERROR_CODE_KEYWORDS = {
    "dtc": ["dtc", "trouble code", "fault code"],
    "sps": ["sps", "programming", "reprogram", "reprogrammed", "calibration", "update"],
    "usb_update": ["usb", "file transfer", "usb programming"],
    "comm_loss_code": ["communication", "loss of comm", "ethernet bus"],
}

ACTION_KEYWORDS = {
    "replaced": ["replaced", "replacement", "install new", "new radio", "new module"],
    "reprogrammed": ["reprogrammed", "reprogram", "programmed", "sps programming", "usb programming"],
    "reset": ["reset", "global reset", "factory settings", "reboot"],
    "inspected": ["inspected", "checked", "tested", "diagnosed", "verified"],
    "updated": ["update", "updated", "ota", "software update"],
}

def extract_entities(text):
    """Return dict of entity lists found in the (raw) text."""
    low = text.lower()
    found = {}
    for cat, kws in [("components", COMPONENT_KEYWORDS),
                     ("symptoms", SYMPTOM_KEYWORDS),
                     ("error_codes", ERROR_CODE_KEYWORDS),
                     ("actions", ACTION_KEYWORDS)]:
        hits = set()
        for label, variants in kws.items():
            for v in variants:
                if v in low:
                    hits.add(label)
                    break
        found[cat] = sorted(hits)
    return found

entity_rows = df_clean["verbatim_combined"].apply(extract_entities)
entity_df = pd.DataFrame(list(entity_rows), index=df_clean.index)

# Expanded tag columns
df_clean["tags_components"] = entity_df["components"].apply(lambda x: "; ".join(x))
df_clean["tags_symptoms"] = entity_df["symptoms"].apply(lambda x: "; ".join(x))
df_clean["tags_error_codes"] = entity_df["error_codes"].apply(lambda x: "; ".join(x))
df_clean["tags_actions"] = entity_df["actions"].apply(lambda x: "; ".join(x))

def all_tags(row):
    parts = []
    for c in ["tags_components","tags_symptoms","tags_error_codes","tags_actions"]:
        v = row[c]
        if v:
            parts.append(v)
    return "; ".join(parts)

df_clean["extracted_tags"] = df_clean.apply(all_tags, axis=1)
df_clean["tag_count"] = df_clean["extracted_tags"].apply(lambda x: 0 if x == "" else len(x.split("; ")))

print("Entity tagging complete.")
print("Sample tagged rows:\n")
print(df_clean[["Event id","CUSTOMER_VERBATIM","tags_components","tags_symptoms","tags_error_codes","tags_actions","extracted_tags"]].head(6).to_string())
print("\nTag coverage: rows with >=1 tag =", int((df_clean["tag_count"]>0).sum()), "of", len(df_clean))


### 2.4 Issue classification — group mined entities into issue categories


In [ ]:

def classify_issue(row):
    """Group mined entities into higher-level issue categories.

    Logic: assembly defects (most specific) > authoritative root-cause code
    (CAUSAL_CD_DESC) > tag-based component/electrical/software signals.
    Using the engineer-determined causal code as a tie-breaker prevents
    reprogramming *actions* from mislabeling hardware *failures* as software.
    """
    causal = str(row.get("CAUSAL_CD_DESC", "")).lower()
    comps = row["tags_components"].lower()
    symps = row["tags_symptoms"].lower()
    errs = row["tags_error_codes"].lower()
    acts = row["tags_actions"].lower()
    combined = " ".join([comps, symps, errs, acts])

    # 1. Assembly Quality: physical/assembly defects (most specific)
    if any(k in combined for k in ["bent pin", "poor contact", "incorrect component", "damaged",
                                   "cracked", "worn", "stripped", "misalignment", "alignment"]):
        return "Assembly Quality"
    # 2. Authoritative root-cause code signals (engineer-determined)
    if any(k in causal for k in ["broken", "damaged/cracked", "worn/stripped", "poor contact",
                                 "module/component-broken"]):
        return "Component Failure"
    if any(k in causal for k in ["shorted", "discharge", "module/component-shorted"]):
        return "Electrical Issue"
    if any(k in causal for k in ["registers incorrectly", "incorrect version"]):
        return "Software Glitch"
    if "no trouble found" in causal:
        return "No Trouble Found"
    # 3. Tag-based: Component Failure (hardware replaced + failure symptom)
    if any(k in comps for k in ["radio", "display", "screen", "module", "amplifier", "antenna",
                                "sd card", "head unit", "infotainment", "tcicm", "telematics"]):
        if (any(k in symps for k in ["black_screen", "no_sound", "freezing", "shorted",
                                     "loss_communication", "wont_load", "blank_on_reverse"])
                or "replaced" in acts):
            return "Component Failure"
    # 4. Tag-based: Electrical Issue (communication / wiring / power)
    if any(k in combined for k in ["loss communication", "lost communication", "no comm",
                                   "shorted", "short", "open circuit", "open condition",
                                   "wiring", "harness", "connector", "ethernet", "ground",
                                   "power", "bcm"]):
        return "Electrical Issue"
    # 5. Tag-based: Software Glitch (reprogram / update / calibration / demo mode)
    if any(k in combined for k in ["sps", "reprogram", "programmed", "calibration",
                                   "software update", "demo mode", "registers incorrectly",
                                   "update", "usb programming"]):
        return "Software Glitch"
    # 6. No Trouble Found
    if any(k in combined for k in ["no trouble found", "unable to duplicate", "adjusted"]):
        return "No Trouble Found"
    return "Other / Unclassified"

df_clean["issue_category"] = df_clean.apply(classify_issue, axis=1)
print("Issue category distribution:\n")
print(df_clean["issue_category"].value_counts().to_string())
df_clean["issue_category"].value_counts().plot(kind="bar", color="#ff7f0e")
plt.title("Issue Category Distribution (from mined entities)")
plt.xlabel("Issue Category"); plt.ylabel("Number of Claims")
plt.xticks(rotation=30, ha="right")
savefig("07_issue_category_distribution.png")

# Cross-tab: issue category vs complaint type
print("\n=== Issue Category x Complaint Type (top) ===")
print(pd.crosstab(df_clean["issue_category"], df_clean["COMPLAINT_CD_DESC"]).to_string())


### 2.5 Export tagged dataset (deliverable dataset)

In [ ]:

tagged_cols = ["Event id","Opened date","MAKE","MODEL","MODLYR","PLANT","BUILD_PLANT_DESC",
               "CAUSAL_CD_DESC","COMPLAINT_CD_DESC","CUSTOMER_VERBATIM","CAUSAL_VERBATIM",
               "CORRECTION_VERBATIM","tags_components","tags_symptoms","tags_error_codes",
               "tags_actions","extracted_tags","tag_count","issue_category",
               "days_build_to_opened","opened_ym"]
df_tagged = df_clean[tagged_cols].copy()

tagged_csv = os.path.join(OUT_DIR, "task2_tagged_dataset.csv")
tagged_xlsx = os.path.join(OUT_DIR, "task2_tagged_dataset.xlsx")
df_tagged.to_csv(tagged_csv, index=False, encoding="utf-8-sig")
df_tagged.to_excel(tagged_xlsx, index=False)
print("Exported tagged dataset:")
print("  CSV :", tagged_csv)
print("  XLSX:", tagged_xlsx)
print("Shape:", df_tagged.shape)


## Step 3 — Clustering / Topic Modelling & Root Cause Analysis


### 3.1 TF-IDF vectorization of cleaned verbatims

In [ ]:

# TF-IDF on cleaned combined verbatim text
tfidf = TfidfVectorizer(max_features=200, ngram_range=(1,2), min_df=3, stop_words="english")
X = tfidf.fit_transform(df_clean["verbatim_clean"])
print("TF-IDF matrix shape:", X.shape)
feature_names = tfidf.get_feature_names_out()

# Save top terms overall
term_scores = np.asarray(X.sum(axis=0)).ravel()
top_terms = pd.DataFrame({"term": feature_names, "tfidf_sum": term_scores})
top_terms = top_terms.sort_values("tfidf_sum", ascending=False)
print("\nTop 25 terms across all verbatims:\n")
print(top_terms.head(25).to_string(index=False))
top_terms.to_csv(os.path.join(OUT_DIR, "step3_top_terms.csv"), index=False)


### 3.2 K-Means clustering on TF-IDF vectors

In [ ]:

# Choose K via silhouette over a range
ks = range(3, 9)
silhouettes = []
inertias = []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))

plt.figure()
plt.plot(list(ks), silhouettes, marker="o", label="Silhouette")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Silhouette score")
plt.title("K-Means Silhouette Score vs K")
plt.legend()
savefig("08_silhouette_vs_k.png")

best_k = int(ks[int(np.argmax(silhouettes))])
print(f"\nBest K by silhouette: {best_k} (silhouette={max(silhouettes):.3f})")

km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = km.fit_predict(X)
df_clean["cluster"] = cluster_labels
print("\nCluster sizes:\n")
print(pd.Series(cluster_labels).value_counts().sort_index().to_string())


### 3.3 Interpret clusters — top terms per cluster = potential root causes

In [ ]:

order_centroids = km.cluster_centers_.argsort()[:, ::-1]
cluster_summary = []
for i in range(best_k):
    top_inds = order_centroids[i, :12]
    top_terms_cluster = [feature_names[idx] for idx in top_inds]
    cluster_mask = df_clean["cluster"] == i
    n = int(cluster_mask.sum())
    # top complaint types & causal codes in this cluster
    top_complaint = df_clean.loc[cluster_mask, "COMPLAINT_CD_DESC"].value_counts().head(2)
    top_causal = df_clean.loc[cluster_mask, "CAUSAL_CD_DESC"].value_counts().head(2)
    top_issue = df_clean.loc[cluster_mask, "issue_category"].value_counts().head(1)
    cluster_summary.append({
        "cluster": i,
        "size": n,
        "pct_of_total": round(n/len(df_clean)*100, 1),
        "top_terms": ", ".join(top_terms_cluster),
        "top_complaint": "; ".join([f"{k}({v})" for k,v in top_complaint.items()]),
        "top_causal": "; ".join([f"{k}({v})" for k,v in top_causal.items()]),
        "top_issue_category": list(top_issue.index)[0] if len(top_issue) else "n/a",
    })
cluster_df = pd.DataFrame(cluster_summary)
print(cluster_df.to_string(index=False))
cluster_df.to_csv(os.path.join(OUT_DIR, "step3_cluster_summary.csv"), index=False)

# Print detailed interpretation
print("\n=== DETAILED CLUSTER INTERPRETATION (potential root causes) ===")
for i in range(best_k):
    cm = df_clean["cluster"] == i
    print(f"\n--- Cluster {i} (n={cm.sum()}) ---")
    print("Top terms:  ", ", ".join([feature_names[idx] for idx in order_centroids[i, :10]]))
    print("Complaint:  ", dict(df_clean.loc[cm, "COMPLAINT_CD_DESC"].value_counts().head(4)))
    print("Root cause: ", dict(df_clean.loc[cm, "CAUSAL_CD_DESC"].value_counts().head(4)))
    print("Issue cat:  ", dict(df_clean.loc[cm, "issue_category"].value_counts().head(3)))
    # representative customer verbatim
    rep = df_clean.loc[cm, "CUSTOMER_VERBATIM"].iloc[0]
    print("Example:    ", rep[:200], "...")


### 3.4 (Optional) Latent Dirichlet Allocation (LDA) topic model

In [ ]:

try:
    from sklearn.decomposition import LatentDirichletAllocation
    n_topics = best_k
    lda = LatentDirichletAllocation(n_components=n_topics, random_state=42, learning_method="batch", max_iter=10)
    lda.fit(X)
    print(f"LDA with {n_topics} topics fitted.")
    for topic_idx, topic in enumerate(lda.components_):
        top_feats = [feature_names[i] for i in topic.argsort()[:-11:-1]]
        print(f"Topic {topic_idx}: " + ", ".join(top_feats))
except Exception as e:
    print("LDA skipped:", e)


### 3.5 Temporal trends — claim frequency over time by issue category

In [ ]:

trend = df_clean.groupby(["opened_ym", "issue_category"]).size().unstack(fill_value=0)
print(trend.to_string())

plt.figure()
trend.plot(kind="area", stacked=True, alpha=0.7)
plt.title("Claim Frequency Over Time by Issue Category")
plt.xlabel("Year-Month"); plt.ylabel("Number of Claims")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Issue Category", bbox_to_anchor=(1.02, 1), loc="upper left")
savefig("09_trend_issue_category_over_time.png", )
plt.tight_layout()
plt.savefig(os.path.join(chart_dir, "09_trend_issue_category_over_time.png"), dpi=150, bbox_inches="tight")
plt.close()
print("Saved chart: 09_trend_issue_category_over_time.png")


## Step 4 — Stakeholder Insights & Recommendations

In [ ]:

print("="*70)
print("KEY INSIGHTS")
print("="*70)
n = len(df_clean)
print(f"1. Total claims analyzed: {n} (all model year 2020 vehicles).")
print(f"2. Date range of claims: {df_clean['Opened date_parsed'].min().date()} to {df_clean['Opened date_parsed'].max().date()}.")

# Top complaint
top_comp = df_clean["COMPLAINT_CD_DESC"].value_counts().iloc[0]
top_comp_name = df_clean["COMPLAINT_CD_DESC"].value_counts().index[0]
print(f"3. Most frequent complaint: '{top_comp_name}' ({top_comp} claims, {top_comp/n*100:.1f}%).")

# Top root cause
top_causal = df_clean["CAUSAL_CD_DESC"].value_counts().iloc[0]
top_causal_name = df_clean["CAUSAL_CD_DESC"].value_counts().index[0]
print(f"4. Most frequent root cause: '{top_causal_name}' ({top_causal} claims, {top_causal/n*100:.1f}%).")

# Top issue category
top_issue = df_clean["issue_category"].value_counts().iloc[0]
top_issue_name = df_clean["issue_category"].value_counts().index[0]
print(f"5. Dominant mined issue category: '{top_issue_name}' ({top_issue} claims, {top_issue/n*100:.1f}%).")

# Make with most claims
top_make = df_clean["MAKE"].value_counts().iloc[0]
top_make_name = df_clean["MAKE"].value_counts().index[0]
print(f"6. Make with most claims: {top_make_name} ({top_make} claims, {top_make/n*100:.1f}%).")

# Avg lag
avg_lag = df_clean["days_build_to_opened"].mean()
print(f"7. Average build->claim lag: {avg_lag:.1f} days (early-life failures indicate manufacturing/assembly issues).")

# Cluster count
print(f"8. {best_k} failure clusters identified via K-Means on TF-IDF text.")

print("\n" + "="*70)
print("ACTIONABLE RECOMMENDATIONS")
print("="*70)
recs = [
 "1. Focus quality remediation on the infotainment RADIO MODULE & DISPLAY — the dominant component failure; "
 "review supplier QC and module firmware calibration at the Silao/Fort Wayne/Flint plants.",
 "2. Address ELECTRICAL / COMMUNICATION issues (loss of communication on Ethernet bus, shorted circuits): "
 "audit wiring harness routing, connector pin tension, and grounding — a leading root cause cluster.",
 "3. Tackle SOFTWARE GLITCHES (demo mode, registers incorrectly, SPS/reprogramming, calibration): "
 "standardize software release validation and OTA update procedures before vehicle launch.",
 "4. Prioritize the top complaint types — Audio, Communication, and Other infotainment issues — which together "
 "account for the majority of claims.",
 "5. Investigate early-life failures (short build->opened lag) at plants with elevated claim rates for "
 "assembly-line process fixes (pin bending, poor contact, incorrect component installation).",
 "6. Use the extracted tags and clusters as a living failure-mode taxonomy for warranty review and "
 "root-cause corrective action (RCCA) meetings.",
 "7. Feed recurring cluster themes back to design engineering for next-model infotainment architecture (e.g., "
 "improved SD-card reader, connector robustness, head-unit thermal/software stability).",
]
for r in recs:
    print(r)


## Save final cleaned + tagged full dataset

In [ ]:

full_out_csv = os.path.join(OUT_DIR, "task2_cleaned_full_dataset.csv")
full_out_xlsx = os.path.join(OUT_DIR, "task2_cleaned_full_dataset.xlsx")
df_clean.to_csv(full_out_csv, index=False, encoding="utf-8-sig")
df_clean.to_excel(full_out_xlsx, index=False)
print("Saved full cleaned dataset:")
print("  CSV :", full_out_csv)
print("  XLSX:", full_out_xlsx)
print("Shape:", df_clean.shape)
print("\nNOTEBOOK ANALYSIS COMPLETE.")
